In [1]:
from pathlib import Path
import csv
import re

### Resultados Categóricos

In [ ]:

# =======================# CONFIGURAÇÃO
# =========================
INPUT_ROOT = Path("final_results\\Categorical\\results")                 # pasta original
OUTPUT_ROOT = Path("categorical_results_transformed")    # nova pasta com a cópia transformada

# Regex para capturar:
# base do dataset + sufixo que será movido para o algoritmo
# Exemplos aceitos:
#   alon-pn-width-2_binary
#   sorlie-pn-freq-4_binary
#   breast-cancer-pn-width-8_nominal
DATASET_PATTERN = re.compile(
    r"^(?P<base>.+)-pn-(?P<suffix>(?:freq|width)-\d+_(?:binary|nominal))$"
)


def transform_row(row: dict) -> dict:
    """
    Exemplo:
      algorithm = 'SSDP+'
      dataset   = 'alon-pn-width-2_binary'
    vira:
      algorithm = 'SSDP+-pn-width-2_binary'
      dataset   = 'alon'
    """
    dataset_value = row["dataset"].strip()
    algorithm_value = row["algorithm"].strip()

    match = DATASET_PATTERN.match(dataset_value)

    # Se não seguir o padrão esperado, mantém a linha como está
    if not match:
        return row

    base_dataset = match.group("base")
    suffix = match.group("suffix")

    row["dataset"] = base_dataset
    row["algorithm"] = f"{algorithm_value}-pn-{suffix}"

    return row


def process_csv_file(input_csv: Path, output_csv: Path) -> tuple[int, int]:
    """
    Processa um CSV e grava a versão transformada no caminho de saída.
    Retorna:
      (total_linhas, linhas_transformadas)
    """
    output_csv.parent.mkdir(parents=True, exist_ok=True)

    total_rows = 0
    transformed_rows = 0

    with input_csv.open("r", encoding="utf-8-sig", newline="") as fin, \
         output_csv.open("w", encoding="utf-8-sig", newline="") as fout:

        reader = csv.DictReader(fin)
        fieldnames = reader.fieldnames

        if fieldnames is None:
            raise ValueError(f"Arquivo sem cabeçalho: {input_csv}")

        writer = csv.DictWriter(fout, fieldnames=fieldnames)
        writer.writeheader()

        for row in reader:
            total_rows += 1

            original_algorithm = row.get("algorithm", "")
            original_dataset = row.get("dataset", "")

            new_row = transform_row(row)

            if (
                new_row.get("algorithm", "") != original_algorithm
                or new_row.get("dataset", "") != original_dataset
            ):
                transformed_rows += 1

            writer.writerow(new_row)

    return total_rows, transformed_rows


def main():
    if not INPUT_ROOT.exists():
        raise FileNotFoundError(f"Pasta de entrada não encontrada: {INPUT_ROOT.resolve()}")

    csv_files = list(INPUT_ROOT.rglob("*.csv"))

    if not csv_files:
        print(f"Nenhum arquivo CSV encontrado em: {INPUT_ROOT.resolve()}")
        return

    total_files = 0
    total_rows = 0
    total_transformed_rows = 0

    for input_csv in csv_files:
        relative_path = input_csv.relative_to(INPUT_ROOT)
        output_csv = OUTPUT_ROOT / relative_path

        rows, transformed = process_csv_file(input_csv, output_csv)

        total_files += 1
        total_rows += rows
        total_transformed_rows += transformed

        print(f"[OK] {input_csv} -> {output_csv} | linhas: {rows} | transformadas: {transformed}")

    print("\n===== RESUMO =====")
    print(f"Arquivos processados: {total_files}")
    print(f"Total de linhas: {total_rows}")
    print(f"Linhas transformadas: {total_transformed_rows}")
    print(f"Saída gravada em: {OUTPUT_ROOT.resolve()}")


if __name__ == "__main__":
    main()

[OK] final_results\Categorical\results\FREQ\QG\simulation-results_qg_freq-2_binary.csv -> results_transformed\FREQ\QG\simulation-results_qg_freq-2_binary.csv | linhas: 570 | transformadas: 570
[OK] final_results\Categorical\results\FREQ\QG\simulation-results_qg_freq-2_nominal.csv -> results_transformed\FREQ\QG\simulation-results_qg_freq-2_nominal.csv | linhas: 570 | transformadas: 570
[OK] final_results\Categorical\results\FREQ\QG\simulation-results_qg_freq-4_binary.csv -> results_transformed\FREQ\QG\simulation-results_qg_freq-4_binary.csv | linhas: 570 | transformadas: 570
[OK] final_results\Categorical\results\FREQ\QG\simulation-results_qg_freq-4_nominal.csv -> results_transformed\FREQ\QG\simulation-results_qg_freq-4_nominal.csv | linhas: 570 | transformadas: 570
[OK] final_results\Categorical\results\FREQ\QG\simulation-results_qg_freq-8_binary.csv -> results_transformed\FREQ\QG\simulation-results_qg_freq-8_binary.csv | linhas: 570 | transformadas: 570
[OK] final_results\Categorical\

### Resultados Numéricos

In [3]:
# ==========================================
# CONFIGURAÇÃO
# ==========================================
INPUT_ROOT = Path("final_results\\Numerical\\results")
OUTPUT_ROOT = Path("numerical_results_transformed")

# Remove o sufixo final "_og_labeled" do dataset
DATASET_PATTERN = re.compile(r"^(?P<base>.+)_og_labeled$")

# Extrai o sufixo do nome do arquivo, ignorando a métrica (wraccn ou qg)
# Exemplos:
#   simulation-results_wraccn_freq-2_nominal.csv -> freq-2_nominal
#   simulation-results_qg_width-8_binary.csv     -> width-8_binary
FILENAME_PATTERN = re.compile(
    r"^simulation-results_(?:wraccn|qg)_(?P<suffix>(?:freq|width)-\d+_(?:binary|nominal))\.csv$",
    re.IGNORECASE
)


def clean_dataset_name(dataset_value: str) -> str:
    """
    Exemplo:
      alon_og_labeled -> alon
      burczynski_og_labeled -> burczynski

    Se não casar com o padrão esperado, mantém como está.
    """
    dataset_value = dataset_value.strip()
    match = DATASET_PATTERN.match(dataset_value)
    if match:
        return match.group("base")
    return dataset_value


def extract_suffix_from_filename(file_path: Path) -> str | None:
    """
    Exemplo:
      simulation-results_wraccn_freq-2_nominal.csv -> freq-2_nominal
    """
    match = FILENAME_PATTERN.match(file_path.name)
    if match:
        return match.group("suffix")
    return None


def transform_row(row: dict, file_suffix: str | None) -> dict:
    """
    Exemplo:
      algorithm = SMHDD_0.05
      dataset   = alon_og_labeled
      file_suffix = freq-2_nominal

    vira:
      algorithm = SMHDD_0.05-freq-2_nominal
      dataset   = alon
    """
    new_row = dict(row)

    # Limpa dataset
    if "dataset" in new_row:
        new_row["dataset"] = clean_dataset_name(new_row["dataset"])

    # Atualiza algorithm com o sufixo do arquivo
    if file_suffix and "algorithm" in new_row:
        algorithm_value = new_row["algorithm"].strip()
        new_row["algorithm"] = f"{algorithm_value}-{file_suffix}"

    return new_row


def process_csv_file(input_csv: Path, output_csv: Path) -> tuple[int, int]:
    """
    Processa um CSV e grava a versão transformada.
    Retorna:
      (total_linhas, linhas_transformadas)
    """
    output_csv.parent.mkdir(parents=True, exist_ok=True)

    total_rows = 0
    transformed_rows = 0
    file_suffix = extract_suffix_from_filename(input_csv)

    with input_csv.open("r", encoding="utf-8-sig", newline="") as fin, \
         output_csv.open("w", encoding="utf-8-sig", newline="") as fout:

        reader = csv.DictReader(fin)
        fieldnames = reader.fieldnames

        if fieldnames is None:
            raise ValueError(f"Arquivo sem cabeçalho: {input_csv}")

        writer = csv.DictWriter(fout, fieldnames=fieldnames)
        writer.writeheader()

        for row in reader:
            total_rows += 1

            original_algorithm = row.get("algorithm", "")
            original_dataset = row.get("dataset", "")

            new_row = transform_row(row, file_suffix)

            if (
                new_row.get("algorithm", "") != original_algorithm
                or new_row.get("dataset", "") != original_dataset
            ):
                transformed_rows += 1

            writer.writerow(new_row)

    return total_rows, transformed_rows


def main():
    if not INPUT_ROOT.exists():
        raise FileNotFoundError(f"Pasta de entrada não encontrada: {INPUT_ROOT.resolve()}")

    csv_files = sorted(INPUT_ROOT.rglob("*.csv"))

    if not csv_files:
        print(f"Nenhum arquivo CSV encontrado em: {INPUT_ROOT.resolve()}")
        return

    total_files = 0
    total_rows = 0
    total_transformed_rows = 0

    for input_csv in csv_files:
        relative_path = input_csv.relative_to(INPUT_ROOT)
        output_csv = OUTPUT_ROOT / relative_path

        rows, transformed = process_csv_file(input_csv, output_csv)

        total_files += 1
        total_rows += rows
        total_transformed_rows += transformed

        print(f"[OK] {input_csv} -> {output_csv} | linhas: {rows} | transformadas: {transformed}")

    print("\n===== RESUMO =====")
    print(f"Arquivos processados: {total_files}")
    print(f"Total de linhas: {total_rows}")
    print(f"Linhas transformadas: {total_transformed_rows}")
    print(f"Saída gravada em: {OUTPUT_ROOT.resolve()}")


if __name__ == "__main__":
    main()

[OK] final_results\Numerical\results\FREQ\QG\simulation-results_qg_freq-2_nominal.csv -> numerical_results_transformed\FREQ\QG\simulation-results_qg_freq-2_nominal.csv | linhas: 190 | transformadas: 190
[OK] final_results\Numerical\results\FREQ\QG\simulation-results_qg_freq-4_binary.csv -> numerical_results_transformed\FREQ\QG\simulation-results_qg_freq-4_binary.csv | linhas: 190 | transformadas: 190
[OK] final_results\Numerical\results\FREQ\QG\simulation-results_qg_freq-4_nominal.csv -> numerical_results_transformed\FREQ\QG\simulation-results_qg_freq-4_nominal.csv | linhas: 190 | transformadas: 190
[OK] final_results\Numerical\results\FREQ\QG\simulation-results_qg_freq-8_binary.csv -> numerical_results_transformed\FREQ\QG\simulation-results_qg_freq-8_binary.csv | linhas: 190 | transformadas: 190
[OK] final_results\Numerical\results\FREQ\QG\simulation-results_qg_freq-8_nominal.csv -> numerical_results_transformed\FREQ\QG\simulation-results_qg_freq-8_nominal.csv | linhas: 190 | transfor

### Generate unified files

In [ ]:
# ==========================================
# CONFIGURAÇÃO
# ==========================================
SOURCE_ROOT = Path("final_results\\Numerical\\results")  # pasta com os CSVs já modificados
OUTPUT_DIR = SOURCE_ROOT                   # onde os arquivos consolidados serão salvos

QG_OUTPUT = OUTPUT_DIR / "numerical_qg_results.csv"
WRACCN_OUTPUT = OUTPUT_DIR / "numerical_wraccn_results.csv"


def list_csvs(folder: Path) -> list[Path]:
    """Retorna todos os CSVs de uma pasta, ordenados pelo nome."""
    if not folder.exists():
        return []
    return sorted(folder.glob("*.csv"))


def merge_csv_files(input_files: list[Path], output_file: Path) -> None:
    """
    Junta vários CSVs em um único arquivo.
    Assume que todos possuem o mesmo cabeçalho.
    """
    if not input_files:
        print(f"[AVISO] Nenhum arquivo encontrado para gerar: {output_file}")
        return

    output_file.parent.mkdir(parents=True, exist_ok=True)

    first_header = None
    total_rows = 0

    with output_file.open("w", encoding="utf-8-sig", newline="") as fout:
        writer = None

        for file_path in input_files:
            with file_path.open("r", encoding="utf-8-sig", newline="") as fin:
                reader = csv.DictReader(fin)

                if reader.fieldnames is None:
                    print(f"[AVISO] Arquivo sem cabeçalho ignorado: {file_path}")
                    continue

                if first_header is None:
                    first_header = reader.fieldnames
                    writer = csv.DictWriter(fout, fieldnames=first_header)
                    writer.writeheader()
                else:
                    if reader.fieldnames != first_header:
                        raise ValueError(
                            f"Cabeçalho diferente encontrado em {file_path}\n"
                            f"Esperado: {first_header}\n"
                            f"Obtido:   {reader.fieldnames}"
                        )

                for row in reader:
                    writer.writerow(row)
                    total_rows += 1

    print(f"[OK] Gerado: {output_file}")
    print(f"     Arquivos usados: {len(input_files)}")
    print(f"     Total de linhas: {total_rows}")


def main():
    # Pastas para QG
    qg_folders = [
        SOURCE_ROOT / "FREQ" / "QG",
        SOURCE_ROOT / "WIDTH" / "QG",
    ]

    # Pastas para WRACCN
    wraccn_folders = [
        SOURCE_ROOT / "FREQ" / "WRACCN",
        SOURCE_ROOT / "WIDTH" / "WRACCN",
    ]

    qg_files = []
    for folder in qg_folders:
        qg_files.extend(list_csvs(folder))

    wraccn_files = []
    for folder in wraccn_folders:
        wraccn_files.extend(list_csvs(folder))

    merge_csv_files(qg_files, QG_OUTPUT)
    merge_csv_files(wraccn_files, WRACCN_OUTPUT)


if __name__ == "__main__":
    main()

[OK] Gerado: numerical_results_transformed\numerical_qg_results.csv
     Arquivos usados: 10
     Total de linhas: 1900
[OK] Gerado: numerical_results_transformed\numerical_wraccn_results.csv
     Arquivos usados: 10
     Total de linhas: 1900


### FRIEDMAN FORMAT

In [6]:
from pathlib import Path
import pandas as pd

# ==========================================
# CONFIGURAÇÃO
# ==========================================
INPUT_CSV = Path("FRIEDMAN READY\\all_wraccn_results.csv")
OUTPUT_CSV = Path("FRIEDMAN READY\\all_wraccn_results-wide.csv")

# coluna que será usada como valor nas células
VALUE_COLUMN = "quality"


def main():
    # Lê o CSV
    df = pd.read_csv(INPUT_CSV, encoding="utf-8-sig")

    # Mantém a ordem original dos algoritmos no arquivo
    algorithm_order = df["algorithm"].drop_duplicates().tolist()

    # Verifica se existe mais de um valor para a mesma combinação
    # (dataset, repetition, algorithm)
    duplicated = df.duplicated(subset=["dataset", "repetition", "algorithm"], keep=False)
    if duplicated.any():
        dup_rows = df.loc[duplicated, ["dataset", "repetition", "algorithm"]]
        raise ValueError(
            "Existem combinações duplicadas de (dataset, repetition, algorithm).\n"
            f"Exemplos:\n{dup_rows.head(10)}"
        )

    # Faz o pivot
    df_wide = df.pivot(
        index=["dataset", "repetition"],
        columns="algorithm",
        values=VALUE_COLUMN
    ).reset_index()

    # Ordena por dataset e repetition
    df_wide = df_wide.sort_values(["dataset", "repetition"]).reset_index(drop=True)

    # Algoritmos na ordem original
    algorithm_columns = [alg for alg in algorithm_order if alg in df_wide.columns]

    # Remove repetition e mantém dataset + algoritmos
    df_wide = df_wide[["dataset", "repetition"] + algorithm_columns]
    df_wide = df_wide.drop(columns=["repetition"])

    # Renomeia dataset
    df_wide = df_wide.rename(columns={"dataset": "Data-set"})

    # Troca "_" por "-" somente nos nomes dos algoritmos
    rename_algorithms = {alg: alg.replace("_", "-") for alg in algorithm_columns}
    df_wide = df_wide.rename(columns=rename_algorithms)

    # Salva
    df_wide.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

    print(f"[OK] Arquivo gerado: {OUTPUT_CSV.resolve()}")
    print(f"Linhas: {len(df_wide)}")
    print(f"Colunas: {len(df_wide.columns)}")


if __name__ == "__main__":
    main()

[OK] Arquivo gerado: C:\Users\giordano.toscano\Documents\SMHDD - PROJETO DEFINITIVO\generate graphics\FRIEDMAN READY\all_wraccn_results-wide.csv
Linhas: 190
Colunas: 47
